### pandas cookbook

All you quickly need to know about pandas

Dr *Gonzalo Martínez Lema*, Dr. *Marija Kekic*

January 2020

In [ ]:
import numpy  as np
import pandas as pd

from IPython.display import display

In [ ]:
np.random.seed(2**30)

# What are Series and dataframes

Series are sequences with arbitrary indexing (usually called label). You may use an accumulative index, which is the default and will always be available anyway

In [ ]:
pd.Series(np.random.uniform(0, 1, 10)) # regular indexing

In [ ]:
pd.Series(np.random.uniform(0, 1, 10), index=list("abcdefghij"))

In [ ]:
pd.Series(np.random.uniform(0, 1, 10), index=pd.date_range('2020-01-01', '2020-01-10'))

A dataframe is a collection of Series with a common index. Both objects are essentially fancy tables.

In [ ]:
pd.DataFrame(np.random.uniform(0, 1, (10, 4)),
             columns = list("ABCD"),
             index   = pd.date_range('2020-01-01', '2020-01-10'))

# Building a Series

In [ ]:
series_0 = pd.Series(data  = np.random.uniform(0, 1, 10)              ,
                     index = pd.date_range('2020-01-01', '2020-01-10'), # optional, default is regular indexing
                     dtype = np.float                                 , # optional, default is inferred
                     name  = "Minutes worked"                         ) # optional, defualt None
series_0

In [ ]:
series_1 = pd.Series(data  = np.random.uniform(0, 1, 10),
                     index = list("ABCDEFGHIJ")         , # optional, default is regular indexing
                     dtype = np.float                   , # optional, default is inferred
                     name  = "Letter affinity"          ) # optional, defualt None
series_1

# Building a dataframe

Probably the easiest manner is to use a dictionary to combine column names with data

In [ ]:
xyz_df_0 = pd.DataFrame(dict(n = np.random.poisson( 4,     10),
                             x = np.random.uniform(-2, -1, 10),
                             y = np.random.uniform( 1,  2, 10),
                             z = np.random.normal ( 0,  1, 10)),
                        index  = list("ABCDEFGHIJ"))             # optional, defaults to regular indexing
xyz_df_0

Let's use a more realistic case and assume the index is something like an event number. We can attach a name to this index as well. Indices can repeat!

In [ ]:
xyz_df_1 = pd.DataFrame(dict(n = np.random.poisson( 4,     10),
                             x = np.random.uniform(-2, -1, 10),
                             y = np.random.uniform( 1,  2, 10),
                             z = np.random.normal ( 0,  1, 10)),
                        index  = [0] * 3 + [1] * 3 + [4] * 4   )

xyz_df_1.index.name = "event" # This will be useful later

xyz_df_1

The same can be done with `pd.Series`

### Exercise 0: Create two or three series. Join them in a dataframe. Each row should correspond to a different time in the same day.

# Displaying a Series/DataFrame

There are a few functionalities to play around with Series and DataFrames. These work for both

In [ ]:
xyz_df_0.head()

In [ ]:
xyz_df_0.tail()

You can know the columns of a DataFrame as

In [ ]:
xyz_df_0.columns

And the shape (nrows, ncols) as

In [ ]:
xyz_df_0.shape

# Statistics

In [ ]:
series_0.describe() # get a series that gives stat info about the data

In [ ]:
xyz_df_0.describe() # get a dataframe that gives stat info about each column

In [ ]:
print(f"""For series_0:
mean = {series_0.mean()}
std  = {series_0.std ()}
min  = {series_0.min ()}
max  = {series_0.max ()}
half = {series_0.quantile(0.5)}
len  = {series_0.count()} = {series_0.size}
""")

Same can be used for a dataframe, producing a Series with the value for each column

In [ ]:
xyz_df_0.mean()

# Indexing

### By position or index/label

In [ ]:
series_1["A"] # by label

This works only for series. Dataframes interpret the argument as a column name

In [ ]:
try:
    xyz_df_0["A"]
except KeyError as error:
    print("KeyError:", error)

In [ ]:
xyz_df_0["x"] # This produces a Series

`[]` is good for playing around, but you don't know if what you are getting is a copy or a view. For serious stuff it is better to use `loc` and `iloc`, which should be as fast (if not faster) and you are guaranteed to get a view (i.e. the same memory positions). This means that modifications to the object you get affect the original one.

In [ ]:
series_1.loc["D"] # by label

In [ ]:
series_1.iloc[3] # by position, i.e. row number

In [ ]:
xyz_df_0.loc["D"] # by label

In [ ]:
xyz_df_0.iloc[3] # by position, i.e. row number

These methods also work with arrays

In [ ]:
series_0.iloc[[1, 5, 6]] # by position

Or slices!

In [ ]:
series_0.iloc[3:6]

For dataframes, rows are columns can be selected simultaneously. Also, the indices or columns do not need to be in the same order as the df

In [ ]:
xyz_df_0.loc[list("EAC"), list("zx")]

Because `loc` returns a view, the original object can be modified. We use a slice range to modify one of the columns of the dataframe

This is important when having to perform many operations on big dataframes, as the copy operations introduce a huge overhead

In [ ]:
xyz_df_2 = xyz_df_0.copy()
xyz_df_2

In [ ]:
xyz_df_2.loc["D":"I", "y"] *= 100

In [ ]:
xyz_df_2

### Exercise 1: copy xyz_df_0 to xyz_df_3 and replace each value of x, y, z, with a normalized value (i.e. subtract mean and divide by std)

### Indexing with default value

In [ ]:
series_1.get("A", np.nan) # Exists, returns value

In [ ]:
series_1.get("dog", np.nan) # Does not exist, returns default

For dataframes we access columns!

In [ ]:
xyz_df_0.get("x") # Exists, returns the column

In [ ]:
xyz_df_0.get("t", 1) # Does not exists, returns default

### Indexing with row-column pairs

In [ ]:
rows    = ["A", "C", "I", "H", "B"]
columns = ['x', 'x', 'y', 'z', 'z']

xyz_df_0.lookup(rows, columns)
# Take x from row A
# Take x from row C
# Take y from row I
# Take z from row H
# Take z from row B

### Fast single-element indexing

This method should only be used for accessing a reduced number of elements. Long iterations using this method are very (very) slow.

In [ ]:
series_1.iat[4], series_1.at["E"] # position and label

In [ ]:
xyz_df_0.iat[4, 2], xyz_df_0.at["E", "z"] # position and label

### Selection by booleans

Essentially the same as for numpy

In [ ]:
mask = series_0 > 0.5
series_0[mask]

In [ ]:
mask  = xyz_df_0.x > -1.5
mask &= xyz_df_0.y <  1.5
mask &= xyz_df_0.z > xyz_df_0.z.quantile(0.5)

In [ ]:
xyz_df_0[mask] # This is a copy! Use loc to avoid it

### Selection by query

This method is not of common use. In words of the developers:
```
A use case for query() is when you have a collection of DataFrame objects that have a subset of column      names (or index levels/names) in common. You can pass the same query to both frames without having to      specify which frame you’re interested in querying
```
Which is not a frequent scenario. Avoid using it unless you find yourself doing something like this.

In [ ]:
quantile = xyz_df_0.z.quantile(0.6)
xyz_df_0.query(f"x < -1.5 & y > 1.5 & z > {quantile}")

### Exercise 2: select from the following dataframe those rows that have exactly 1 of the values in the provided list

In [ ]:
df_with_ints = pd.DataFrame(dict(x = np.random.poisson(5, 10),
                                 y = np.random.poisson(2, 10),
                                 z = np.random.poisson(8, 10)))

list_of_ints = [1, 4, 5, 9]
df_with_ints

### Useful functions for selections

In [ ]:
xyz_df_0.n.isin([4, 5]) # True if value is in the provided list, False otherwise

In [ ]:
series_2 = series_0.copy()
series_2.iloc[3::3] = np.nan
series_2.isna() # True if value is a NaN They can also be dropped with series.dropna()

In [ ]:
xyz_df_0.n.duplicated()
# True if the is duplicated. Default is to mark as duplicated only 2nd, 3rd, ... occurrences
# Check docs for other options

# Sampling

We can take samples from a series

In [ ]:
series_0.sample()

Or many at a time

In [ ]:
xyz_df_0.sample(3)

Or a fraction of the total

In [ ]:
series_1.sample(frac=0.334)

# Combining data

### Row-wise

In [ ]:
first_events = xyz_df_0.iloc[  :3]
last_events  = xyz_df_1.iloc[-3: ]
display(first_events, last_events)

In [ ]:
pd.concat([first_events, last_events]) # New dataframe

The same can be achieved with `append`

In [ ]:
first_events.append(last_events) # new dataframe!

You may also append (or concat) a series

In [ ]:
first_events.append(xyz_df_1.iloc[4]) # new dataframe!

### Column-wise

In [ ]:
first_columns = xyz_df_1.iloc[:,   :1]
last_columns  = xyz_df_1.iloc[:, -2: ]
display(first_columns, last_columns)

In [ ]:
pd.concat([first_columns, last_columns], axis=1) # New dataframe

If there are repeated columns the default behaviour is to keep both (`join="outer"`). Use `join="inner"` to keep the intersection.

You may also concatenate a series to a dataframe

In [ ]:
pd.concat([first_columns, xyz_df_1.y], axis=1)

### Exercise 3: join `xyz_df_0` and `xyz_df_2` using `join="outer"`

In [ ]:
display(xyz_df_0, xyz_df_2)

# Grouping data

Usually we want to perform operations to groups of data. The most common cases is "I want to do this event by event" or "peak by peak". The easiest way is to use `groupby`.

In [ ]:
display(xyz_df_1)
groups = xyz_df_1.groupby("event")

k = 0
for event_no, subdf in groups:
    display(subdf)

We can also group by more than one label

In [ ]:
groups = xyz_df_1.groupby(["event", "n"])

k = 0
for (event_no, peak_no), subdf in groups:
    display(subdf)

If we are interested in a specific group, not all of them, we can also get them

In [ ]:
groups.get_group((1, 4)) # event=1, n=5

# Applying operations on each group

We can get generic properties of each grup easily

In [ ]:
groups = xyz_df_1.groupby("event")
groups.size() # Count rows in each group

In [ ]:
groups.mean()

We can get it for a specific column as well (faster if you don't need to apply the same operation onto many columns)

In [ ]:
groups.x.mean() # groups["x"] would also work

We can also produce a new dataframe with this information with `aggregate`

In [ ]:
groups.aggregate(np.sum) # Produce a dataframe indexed with event and summing all entries in each column

We may want to apply different operations to different columns

In [ ]:
groups.aggregate(dict(n=np.sum, x=np.mean, y="min", z="first"))
# Same, but only with the specified operation to each column

You may also want to get many properties of the same data

In [ ]:
groups.x.aggregate([np.mean, np.std, "count"]) # produce a df with these columns for each group

It also works for a dataframe, but it gets messy

In [ ]:
groups.aggregate(["min", "max"]) # Produce dataframe with two column levels

You don't like these names? No problem!

In [ ]:
groups.aggregate(dict(n=np.sum, x=np.mean, y="min", z="first"))\
      .rename(columns = dict(n = "sum_n", x = "av_x", y = "min_y", z = "first_z"))

Naturally, you can use your functions. They must take an array (dataframe) and return a scalar (series). Warning, this might be super slow!

In [ ]:
def weird_function(x):
    return -1 if np.sum(x) < -2 else +1

In [ ]:
groups.z.aggregate([lambda x: np.max(x) - np.min(x), weird_function, np.sum])

This however, creates a problem when we try to add this group-dependent information to the original dataframe:

In [ ]:
sum_n = xyz_df_1.groupby("event").n.aggregate("sum") # same as xyz_df_1.groupby("event").n.sum()
xyz_df_1.shape[0], sum_n.shape[0]

This can be avoided using transform:

In [ ]:
sum_n = xyz_df_1.groupby("event").n.transform("sum") # same as xyz_df_1.groupby("event").n.sum()
xyz_df_1.shape[0], sum_n.shape[0]

This way we can add this information to the original dataframe

In [ ]:
xyz_df_1.loc[:, "sum_n"] = xyz_df_1.groupby("event").n.transform(np.sum)

In [ ]:
xyz_df_1

The function used in transform can either:
- Return a single value, in which case, the value is broadcasted to match the size of the group
- Return an array of values of the same size of the group. Each value get's assigned to the corresponding entry

In [ ]:
xyz_df_1.groupby("event").z.transform(lambda v: v - v.min())

### *** Warning
* Using functions that are not "standard" increases the execution time. Avoid it whenever possible.
* This is because the standard ones are optimized by Cython. If you implement your custom function in cython (or if you cythonize it inline) it should be as fast

A good use of this method is to replace NaN values with, for example, the average value of each event.
### Exercise 4: Use `transform` to do so with the following dataframe.

In [ ]:
event = np.repeat(np.arange(5), 4)
x     = np.random.uniform(-1, 1, 20)
y     = np.random.uniform(-1, 1, 20)
x[[0, 5, 10, 15]] = np.nan
y[[4, 7, 10, 12]] = np.nan
df_with_nans = pd.DataFrame(dict(x=x, y=y), index=event)
df_with_nans

# Filtering groups

In [ ]:
xyz_df_1.groupby("event").filter(lambda g: np.any(g.z) > 0)
# produces a new df by concatenating the groups that satisfy the condition concatenated

# Apply / applymap / map

- `apply` is used to generate new values from a dataframe from a function column- or row-wise. The function may or may not reduce the data
- `applymap` same but element-wise. It cannot reduce the data
- `map` is used to translate values

In [ ]:
xyz_df_1.apply(lambda x: np.sqrt(np.sum(x**2)), axis=0) # default

In [ ]:
xyz_df_1.apply(lambda x: np.sqrt(np.sum(x**2)), axis=1)

In [ ]:
xyz_df_1.applymap(lambda x: np.sqrt(np.abs(x)))

In [ ]:
mapping = {0: -1, 1: 10, 2: np.pi, 3: 44, 4:123, 5:1e3} # Missing values go missing!
xyz_df_1.loc[:, ["n"]].applymap(mapping.get) # Notice the list in `loc`: If forces the result to be a dataframe

In [ ]:
mapping = {0: -1, 1: 10, 2: np.pi, 3: 44, 4:123, 5:1e3}
xyz_df_1.n.map(mapping)

In [ ]:
min_z = xyz_df_1.z.min()
xyz_df_1.z.map(lambda zz: zz - min_z)

### Exercise 5: create a dataframe with the average position of each event in XYZ. Then compute the R and Phi positions of these average values.

### Exercise 6: Repeat the previous exercise but adding those values to the original df